In [0]:

# NOTEBOOK 3: CAPA GOLD (ANALÍTICA)
# Proyecto: Integración de Ventas y Catálogo

# 1. CONFIGURACIÓN DE PARÁMETROS Y AUTENTICACIÓN
v_storage = "datalaketpint1"
v_key = "AZURE_STORAGE_KEY_AQUI"

spark.conf.set(
    f"fs.azure.account.key.{v_storage}.dfs.core.windows.net",
    v_key
)

from pyspark.sql.functions import col, sum, round

# 2. DEFINICIÓN DE RUTAS
path_ventas_silver = f"abfss://silver@{v_storage}.dfs.core.windows.net/ventas_limpias"
path_productos_silver = f"abfss://silver@{v_storage}.dfs.core.windows.net/productos_limpios"
path_gold = f"abfss://gold@{v_storage}.dfs.core.windows.net/reporte_final"

print("Cargando datos desde la capa Silver...")

# 3. LECTURA DE DATOS (DELTA FORMAT)
try:
    df_ventas = spark.read.format("delta").load(path_ventas_silver)
    df_productos = spark.read.format("delta").load(path_productos_silver)
except Exception as e:
    print(f"Error al leer datos: {e}. Verificá que los Notebooks 1 y 2 hayan corrido con éxito.")

# 4. TRANSFORMACIÓN ANALÍTICA (JOIN Y ENRIQUECIMIENTO)
# Unimos ventas con productos para tener el nombre y el precio en una sola tabla
df_enriquecido = df_ventas.join(
    df_productos, 
    df_ventas["ID_Producto"] == df_productos["id"], 
    "inner"
)

# 5. CÁLCULO DE MÉTRICAS (VENTA TOTAL)
df_final = df_enriquecido.select(
    col("ID_Venta"),
    col("title").alias("Producto_Nombre"),
    col("category").alias("Categoria"),
    col("Cantidad"),
    col("price").alias("Precio_Unitario"),
    (col("Cantidad") * col("price")).alias("Venta_Total")
)

# 6. LIMPIEZA DE DESTINO Y GUARDADO EN GOLD
print(f"Escribiendo resultado final en: {path_gold}")

# Borramos carpeta anterior si existe para evitar conflictos de esquema
dbutils.fs.rm(path_gold, recurse=True)

# Guardamos como tabla Delta
df_final.write.format("delta").mode("overwrite").save(path_gold)

# 7. VISUALIZACIÓN 
print("¡PIPELINE FINALIZADO CON ÉXITO!")
print("Ranking de Ventas por Categoría:")

# Mostramos una agregación rápida para la visualización de Databricks
resumen_gerencial = df_final.groupBy("Categoria") \
    .agg(round(sum("Venta_Total"), 2).alias("Ingresos_Totales")) \
    .orderBy(col("Ingresos_Totales").desc())

display(resumen_gerencial)

Cargando datos desde la capa Silver...
Escribiendo resultado final en: abfss://gold@datalaketpint1.dfs.core.windows.net/reporte_final
------------------------------------------------------------
¡PIPELINE FINALIZADO CON ÉXITO!
Ranking de Ventas por Categoría:
------------------------------------------------------------


Categoria,Ingresos_Totales
electronics,477947.83
jewelery,265175.87
men's clothing,55860.78
women's clothing,40919.1


Databricks visualization. Run in Databricks to view.